# EEGNet Modeling v2


## 1. Load data

In [2]:
from google.colab import drive
import pickle
import numpy as np
import pandas as pd
from scipy.signal import welch
from scipy import stats
from sklearn.model_selection import GroupKFold, train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import copy
import json

drive.mount('/content/drive')

# from the new preprocessing notebook -- already includes epoch_session_ids and the global amplitude filter
pkl_path = '/content/drive/MyDrive/processed_eeg_dataset3.pkl'

with open(pkl_path, 'rb') as f:
    data = pickle.load(f)

epochs = data['epochs']
epoch_labels = data['epoch_labels']
epoch_trial_ids = data['epoch_trial_ids']
epoch_session_ids = data['epoch_session_ids']

channel_cols = ['FZ', 'C3', 'CZ', 'C4']
label_map = {'Left': 0, 'Right': 1}

y_all = np.array([label_map[l] for l in epoch_labels])
trial_ids_all = np.array(epoch_trial_ids)
session_ids_all = np.array(epoch_session_ids)

print(f"Epochs: {len(epochs)}")
print(f"Unique trials: {len(set(trial_ids_all))}")
print(f"Unique sessions: {len(set(session_ids_all))}")
print(f"Label distribution: {pd.Series(epoch_labels).value_counts().to_dict()}")
print(f"Epoch shape: {epochs[0].shape}, columns: {epochs[0].columns.tolist()}")

Mounted at /content/drive
Epochs: 4384
Unique trials: 1782
Unique sessions: 40
Label distribution: {'Left': 2203, 'Right': 2181}
Epoch shape: (500, 5), columns: ['Time', 'FZ', 'C3', 'CZ', 'C4']


## 2. Outer split: session-based GroupKFold

Splitting by `epoch_trial_ids` would let the same session show up in both train and test, since one session contains many trials. Splitting by `epoch_session_ids` guarantees test sessions are entirely unseen during training -- this is the split that actually tests generalization to new sessions.

In [3]:
FOLD = 0  # which of the 5 outer folds to use for the main run below

gkf_outer = GroupKFold(n_splits=5)
outer_folds = list(gkf_outer.split(np.zeros(len(y_all)), y_all, groups=session_ids_all))

train_idx_full, test_idx = outer_folds[FOLD]

print(f"Fold {FOLD}")
print(f"Train+val epochs: {len(train_idx_full)}, Test epochs: {len(test_idx)}")
print(f"Train+val sessions: {len(set(session_ids_all[train_idx_full]))}, Test sessions: {len(set(session_ids_all[test_idx]))}")
print(f"Session overlap with test (should be 0): {len(set(session_ids_all[train_idx_full]) & set(session_ids_all[test_idx]))}")

Fold 0
Train+val epochs: 3505, Test epochs: 879
Train+val sessions: 32, Test sessions: 8
Session overlap with test (should be 0): 0


## 3. Inner split: carve a validation set out of training sessions

This is the piece the old notebook was missing. Splitting `train_idx_full`'s sessions again (not epochs directly, to avoid session leakage here too) gives a validation set that mimics test conditions -- sessions the model has never seen -- so early stopping decisions don't touch the actual test set.

In [4]:
train_sessions_full = sorted(set(session_ids_all[train_idx_full]))
fit_sessions, val_sessions = train_test_split(train_sessions_full, test_size=0.2, random_state=42)
fit_sessions, val_sessions = set(fit_sessions), set(val_sessions)

fit_idx = np.array([i for i in train_idx_full if session_ids_all[i] in fit_sessions])
val_idx = np.array([i for i in train_idx_full if session_ids_all[i] in val_sessions])

print(f"Fit sessions: {len(fit_sessions)}, Val sessions: {len(val_sessions)}")
print(f"Fit epochs: {len(fit_idx)}, Val epochs: {len(val_idx)}")
print(f"Fit/Val session overlap (should be 0): {len(fit_sessions & val_sessions)}")

Fit sessions: 25, Val sessions: 7
Fit epochs: 2788, Val epochs: 717
Fit/Val session overlap (should be 0): 0


## 4. Per-session normalization (fallback scaler fixed)

Fits one `StandardScaler` per session using only `fit_idx`. Any session not seen in `fit_idx` (val and test sessions, by construction) falls back to a single scaler fit on all of `fit_idx` -- computed once, not recomputed per epoch, which is what caused the 7-minute runtime in the old notebook.

In [5]:
def fit_transform_by_session(fit_idx, apply_idx_list, epochs, channel_cols, session_ids):
    """
    Fits one StandardScaler per session using only fit_idx epochs.
    Transforms each index set in apply_idx_list using that session's scaler
    if it was seen during fitting, otherwise a single fallback scaler fit
    on all of fit_idx (computed once).
    Returns (list_of_scaled_epoch_lists, session_scalers).
    """
    session_scalers = {}
    fit_sessions_seen = set(session_ids[i] for i in fit_idx)

    for sess in fit_sessions_seen:
        sess_positions = [i for i in fit_idx if session_ids[i] == sess]
        stack = np.vstack([epochs[i][channel_cols].values for i in sess_positions])
        session_scalers[sess] = StandardScaler().fit(stack)

    fallback_scaler = StandardScaler().fit(
        np.vstack([epochs[i][channel_cols].values for i in fit_idx])
    )

    results = []
    for idx_set in apply_idx_list:
        scaled = []
        for i in idx_set:
            sess = session_ids[i]
            scaler = session_scalers.get(sess, fallback_scaler)
            scaled.append(scaler.transform(epochs[i][channel_cols].values))
        results.append(scaled)

    return results, session_scalers


def reshape_for_eegnet(epoch_list):
    # each epoch: (500, 4) -> transpose to (4, 500) -> add dummy dim -> (1, 4, 500)
    reshaped = np.array([ep.T for ep in epoch_list])
    reshaped = reshaped[:, np.newaxis, :, :]
    return reshaped

In [6]:
(fit_scaled, val_scaled, test_scaled), session_scalers = fit_transform_by_session(
    fit_idx, [fit_idx, val_idx, test_idx], epochs, channel_cols, session_ids_all
)

X_fit = reshape_for_eegnet(fit_scaled)
X_val = reshape_for_eegnet(val_scaled)
X_test = reshape_for_eegnet(test_scaled)

y_fit = y_all[fit_idx]
y_val = y_all[val_idx]
y_test = y_all[test_idx]

print(f"X_fit: {X_fit.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")
print(f"y_fit balance: {np.bincount(y_fit)}")
print(f"y_val balance: {np.bincount(y_val)}")
print(f"y_test balance: {np.bincount(y_test)}")

X_fit: (2788, 1, 4, 500), X_val: (717, 1, 4, 500), X_test: (879, 1, 4, 500)
y_fit balance: [1412 1376]
y_val balance: [343 374]
y_test balance: [448 431]


## 5. EEGNet architecture

Same architecture as before -- the overfit sanity check already proved this has enough capacity, so no changes needed here.

In [7]:
class EEGNet(nn.Module):
    def __init__(self, n_channels=4, n_samples=500, n_classes=2, dropout=0.5):
        super(EEGNet, self).__init__()

        F1 = 8
        D = 2
        F2 = F1 * D
        kernel_length = 64

        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, kernel_length), padding=(0, kernel_length // 2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, F1 * D, kernel_size=(n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, kernel_size=(1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(dropout)
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            out = self.block2(self.block1(dummy))
            flat_size = out.view(1, -1).shape[1]

        self.classifier = nn.Linear(flat_size, n_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# quick shape sanity check
model = EEGNet(n_channels=4, n_samples=X_fit.shape[-1], dropout=0.5)
dummy_out = model(torch.tensor(X_fit[:4], dtype=torch.float32))
print(f"Output shape: {dummy_out.shape}")  # should be (4, 2)

Output shape: torch.Size([4, 2])


## 6. Training with proper early stopping (uses val, never test)

In [8]:
def train_eegnet(X_train, y_train, X_val, y_val, n_channels=4, n_samples=500,
                  epochs=150, patience=30, batch_size=64, lr=5e-4, dropout=0.25, seed=0, verbose=True):
    torch.manual_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = EEGNet(n_channels=n_channels, n_samples=n_samples, dropout=dropout).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                              torch.tensor(y_train, dtype=torch.long))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val_t = torch.tensor(y_val, dtype=torch.long).to(device)

    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
            train_correct += (outputs.argmax(1) == y_batch).sum().item()
            train_total += X_batch.size(0)
        train_loss /= train_total
        train_acc = train_correct / train_total

        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_t)
            val_loss = criterion(val_outputs, y_val_t).item()
            val_acc = (val_outputs.argmax(1) == y_val_t).float().mean().item()

        if verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
            print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
                  f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_model_state)
    return model, best_val_loss

## 7. IMPORTANT -- run the real-vs-shuffled check before trusting any accuracy below

This is the diagnostic that overturned the earlier per-session-signal theory on the band-power baseline. Run it here for EEGNet, across multiple folds and seeds, using only fit/val (test stays untouched). Only if real labels clearly and consistently beat shuffled labels does the accuracy number in section 8 mean anything.

In [9]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

def quick_train_eegnet(X_train, y_train, X_eval, y_eval, n_epochs=40, seed=0, dropout=0.5):
    torch.manual_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = EEGNet(n_channels=4, n_samples=X_train.shape[-1], n_classes=2, dropout=dropout).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.long).to(device)
    X_ev = torch.tensor(X_eval, dtype=torch.float32).to(device)
    y_ev = torch.tensor(y_eval, dtype=torch.long).to(device)

    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        out = model(X_tr)
        loss = criterion(out, y_tr)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        eval_acc = (model(X_ev).argmax(1) == y_ev).float().mean().item()
    return eval_acc


n_folds_to_test = 3   # set to 1 first if you just want a quick sanity check
n_seeds = 5           # set to 2 first if you just want a quick sanity check

real_scores = []
shuffled_scores = []

for fold_i in range(n_folds_to_test):
    tr_idx_full, te_idx = outer_folds[fold_i]
    tr_sessions_full = sorted(set(session_ids_all[tr_idx_full]))
    fit_s, val_s = train_test_split(tr_sessions_full, test_size=0.2, random_state=42)
    fit_s, val_s = set(fit_s), set(val_s)

    fit_i = np.array([i for i in tr_idx_full if session_ids_all[i] in fit_s])
    val_i = np.array([i for i in tr_idx_full if session_ids_all[i] in val_s])

    (fit_sc, val_sc), _ = fit_transform_by_session(fit_i, [fit_i, val_i], epochs, channel_cols, session_ids_all)
    X_fit_f = reshape_for_eegnet(fit_sc)
    X_val_f = reshape_for_eegnet(val_sc)
    y_fit_f = y_all[fit_i]
    y_val_f = y_all[val_i]

    print(f"Fold {fold_i}: fit={len(fit_i)}, val={len(val_i)}")

    for seed in range(n_seeds):
        real_acc = quick_train_eegnet(X_fit_f, y_fit_f, X_val_f, y_val_f, seed=seed)
        real_scores.append(real_acc)

        rng = np.random.RandomState(seed)
        y_fit_shuf = rng.permutation(y_fit_f)
        y_val_shuf = rng.permutation(y_val_f)
        shuf_acc = quick_train_eegnet(X_fit_f, y_fit_shuf, X_val_f, y_val_shuf, seed=seed)
        shuffled_scores.append(shuf_acc)

        print(f"  seed {seed}: real={real_acc:.3f}, shuffled={shuf_acc:.3f}")

    print(f"Fold {fold_i} done\n")

real_scores = np.array(real_scores)
shuffled_scores = np.array(shuffled_scores)

print(f"Real     - mean: {real_scores.mean():.3f}, std: {real_scores.std():.3f}")
print(f"Shuffled - mean: {shuffled_scores.mean():.3f}, std: {shuffled_scores.std():.3f}")

t_stat, p_val = stats.ttest_ind(real_scores, shuffled_scores)
print(f"\nt={t_stat:.2f}, p={p_val:.4f}")
print("\nIf p is well below 0.05 and real's mean is clearly above shuffled's, proceed to section 8.")
print("If they overlap heavily, the accuracy in section 8 should not be trusted as evidence of real signal.")

CUDA available: True
Device: cuda
Fold 0: fit=2788, val=717
  seed 0: real=0.510, shuffled=0.488
  seed 1: real=0.523, shuffled=0.495
  seed 2: real=0.513, shuffled=0.484
  seed 3: real=0.527, shuffled=0.516
  seed 4: real=0.478, shuffled=0.490
Fold 0 done

Fold 1: fit=2828, val=675
  seed 0: real=0.498, shuffled=0.486
  seed 1: real=0.541, shuffled=0.477
  seed 2: real=0.502, shuffled=0.489
  seed 3: real=0.533, shuffled=0.507
  seed 4: real=0.489, shuffled=0.486
Fold 1 done

Fold 2: fit=2736, val=767
  seed 0: real=0.512, shuffled=0.533
  seed 1: real=0.508, shuffled=0.473
  seed 2: real=0.518, shuffled=0.519
  seed 3: real=0.506, shuffled=0.499
  seed 4: real=0.486, shuffled=0.532
Fold 2 done

Real     - mean: 0.510, std: 0.017
Shuffled - mean: 0.498, std: 0.018

t=1.72, p=0.0959

If p is well below 0.05 and real's mean is clearly above shuffled's, proceed to section 8.
If they overlap heavily, the accuracy in section 8 should not be trusted as evidence of real signal.


## 8. Final training run and held-out test evaluation

Only meaningful once section 7 shows real labels beating shuffled labels. The test set here has not been touched by any decision so far (not scaling, not early stopping, not model selection).

In [10]:
model_fold0, best_val_loss = train_eegnet(X_fit, y_fit, X_val, y_val)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_fold0.eval()
X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)
with torch.no_grad():
    test_acc = (model_fold0(X_test_t).argmax(1) == y_test_t).float().mean().item()

print(f"\nFinal held-out test accuracy (fold {FOLD}, unseen sessions): {test_acc:.4f}")

Epoch 1: train_loss=0.7017, train_acc=0.5222, val_loss=0.7016, val_acc=0.5021
Epoch 10: train_loss=0.6874, train_acc=0.5391, val_loss=0.7057, val_acc=0.5105
Epoch 20: train_loss=0.6770, train_acc=0.5588, val_loss=0.7106, val_acc=0.5105
Epoch 30: train_loss=0.6678, train_acc=0.5933, val_loss=0.7148, val_acc=0.5077
Early stopping at epoch 33

Final held-out test accuracy (fold 0, unseen sessions): 0.5188


## 9. Save model

In [11]:
torch.save(model_fold0.state_dict(), "/content/eegnet_model_v2.pth")

model_config = {
    "n_channels": 4,
    "n_samples": int(X_fit.shape[-1]),
    "n_classes": 2,
    "dropout": 0.25,
    "channel_order": channel_cols,
    "label_map": label_map,
    "fold": FOLD,
    "test_accuracy": test_acc
}

with open("/content/eegnet_config_v2.json", "w") as f:
    json.dump(model_config, f, indent=2)

print("Saved model weights to eegnet_model_v2.pth")
print("Saved config to eegnet_config_v2.json")

Saved model weights to eegnet_model_v2.pth
Saved config to eegnet_config_v2.json


LSTM

In [12]:
# LSTM wants (batch, seq_len, n_channels) — the scaled epochs from section 4 are
# already (500, 4), so no transpose needed here, unlike reshape_for_eegnet
X_fit_lstm = np.array(fit_scaled)    # (n, 500, 4)
X_val_lstm = np.array(val_scaled)
X_test_lstm = np.array(test_scaled)

print(f"X_fit_lstm: {X_fit_lstm.shape}, X_val_lstm: {X_val_lstm.shape}, X_test_lstm: {X_test_lstm.shape}")

X_fit_lstm: (2788, 500, 4), X_val_lstm: (717, 500, 4), X_test_lstm: (879, 500, 4)


In [13]:
class SimpleLSTM(nn.Module):
    def __init__(self, n_channels=4, hidden_size=16, n_classes=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_channels, hidden_size=hidden_size,
                             num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, n_classes)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)   # h_n: (num_layers, batch, hidden_size)
        last_hidden = self.dropout(h_n[-1])
        return self.classifier(last_hidden)

# shape sanity check
m = SimpleLSTM()
out = m(torch.tensor(X_fit_lstm[:4], dtype=torch.float32))
print(f"SimpleLSTM output shape: {out.shape}")  # should be (4, 2)

SimpleLSTM output shape: torch.Size([4, 2])


In [14]:
class ComplexLSTM(nn.Module):
    def __init__(self, n_channels=4, hidden_size=64, num_layers=3, n_classes=2, dropout=0.5):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_channels, hidden_size=hidden_size, num_layers=num_layers,
                             batch_first=True, dropout=dropout, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_size * 2, hidden_size)  # *2 for bidirectional
        self.relu = nn.ReLU()
        self.classifier = nn.Linear(hidden_size, n_classes)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        last_fwd = h_n[-2]   # last layer, forward direction
        last_bwd = h_n[-1]   # last layer, backward direction
        combined = self.dropout(torch.cat([last_fwd, last_bwd], dim=1))
        x = self.relu(self.fc1(combined))
        return self.classifier(x)

# shape sanity check
m = ComplexLSTM()
out = m(torch.tensor(X_fit_lstm[:4], dtype=torch.float32))
print(f"ComplexLSTM output shape: {out.shape}")  # should be (4, 2)

ComplexLSTM output shape: torch.Size([4, 2])


In [15]:
def train_generic_model(model_class, model_kwargs, X_train, y_train, X_val, y_val,
                         epochs=150, patience=30, batch_size=64, lr=5e-4, seed=0, verbose=True):
    torch.manual_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model = model_class(**model_kwargs).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                              torch.tensor(y_train, dtype=torch.long))
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val_t = torch.tensor(y_val, dtype=torch.long).to(device)

    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
            train_correct += (outputs.argmax(1) == y_batch).sum().item()
            train_total += X_batch.size(0)
        train_loss /= train_total
        train_acc = train_correct / train_total

        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_t)
            val_loss = criterion(val_outputs, y_val_t).item()
            val_acc = (val_outputs.argmax(1) == y_val_t).float().mean().item()

        if verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
            print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, "
                  f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                if verbose:
                    print(f"Early stopping at epoch {epoch+1}")
                break

    model.load_state_dict(best_model_state)
    return model, best_val_loss

In [16]:
print("=== Training Simple LSTM (low lr) ===")
simple_lstm, simple_val_loss = train_generic_model(
    SimpleLSTM, {"n_channels": 4, "hidden_size": 16, "n_classes": 2, "dropout": 0.3},
    X_fit_lstm, y_fit, X_val_lstm, y_val,
    lr=1e-4
)

print("\n=== Training Complex LSTM (multi-layer, bidirectional) ===")
complex_lstm, complex_val_loss = train_generic_model(
    ComplexLSTM, {"n_channels": 4, "hidden_size": 64, "num_layers": 3, "n_classes": 2, "dropout": 0.5},
    X_fit_lstm, y_fit, X_val_lstm, y_val,
    lr=5e-4
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def eval_test_acc(model, X_test_arr, y_test_arr):
    model.eval()
    X_t = torch.tensor(X_test_arr, dtype=torch.float32).to(device)
    y_t = torch.tensor(y_test_arr, dtype=torch.long).to(device)
    with torch.no_grad():
        return (model(X_t).argmax(1) == y_t).float().mean().item()

eegnet_test_acc = eval_test_acc(model_fold0, X_test, y_test)          # EEGNet from section 8, shape (n,1,4,500)
simple_lstm_test_acc = eval_test_acc(simple_lstm, X_test_lstm, y_test)
complex_lstm_test_acc = eval_test_acc(complex_lstm, X_test_lstm, y_test)

print("\n=== Final comparison (held-out test, unseen sessions) ===")
print(f"EEGNet       : {eegnet_test_acc:.4f}")
print(f"Simple LSTM  : {simple_lstm_test_acc:.4f}")
print(f"Complex LSTM : {complex_lstm_test_acc:.4f}")

=== Training Simple LSTM (low lr) ===
Epoch 1: train_loss=0.6992, train_acc=0.4889, val_loss=0.6931, val_acc=0.5230
Epoch 10: train_loss=0.6951, train_acc=0.4968, val_loss=0.6927, val_acc=0.5328
Epoch 20: train_loss=0.6917, train_acc=0.5176, val_loss=0.6933, val_acc=0.5188
Epoch 30: train_loss=0.6929, train_acc=0.5118, val_loss=0.6936, val_acc=0.5049
Early stopping at epoch 36

=== Training Complex LSTM (multi-layer, bidirectional) ===
Epoch 1: train_loss=0.6933, train_acc=0.5014, val_loss=0.6959, val_acc=0.4784
Epoch 10: train_loss=0.6867, train_acc=0.5412, val_loss=0.7007, val_acc=0.4993
Epoch 20: train_loss=0.6828, train_acc=0.5470, val_loss=0.7049, val_acc=0.5077
Epoch 30: train_loss=0.6712, train_acc=0.5642, val_loss=0.7334, val_acc=0.4756
Early stopping at epoch 35

=== Final comparison (held-out test, unseen sessions) ===
EEGNet       : 0.5188
Simple LSTM  : 0.4653
Complex LSTM : 0.5245
